# Lab 01 - Sentiment Analysis Using Traditional Machine Learning

### 1. Download and load dataset

In [9]:
import pandas as pd

amazon = pd.read_csv(r"C:\Users\Owner\Downloads\LLM-Practical-Portfolio\Lab-01-Machine-Learning\data\sentiment+labelled+sentences\sentiment labelled sentences\amazon_cells_labelled.txt",
                       sep="\t", header=None, names=["text", "sentiment"])

imdb = pd.read_csv(r"C:\Users\Owner\Downloads\LLM-Practical-Portfolio\Lab-01-Machine-Learning\data\sentiment+labelled+sentences\sentiment labelled sentences\imdb_labelled.txt",
                       sep="\t", header=None, names=["text", "sentiment"])

yelp = pd.read_csv(r"C:\Users\Owner\Downloads\LLM-Practical-Portfolio\Lab-01-Machine-Learning\data\sentiment+labelled+sentences\sentiment labelled sentences\yelp_labelled.txt",
                       sep="\t", header=None, names=["text", "sentiment"])

df = pd.concat([amazon, imdb, yelp], ignore_index=True)

df.head()


,text,sentiment
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


In [2]:
df.shape


(2748, 2)

### 2. Perform basic text preprocessing

In [10]:

import re

#rename column 
df = df.rename(columns={"text": "review"})

#Check and remove if there is any incomplete rows, check label values
df = df.dropna(subset=["review", "sentiment"]).copy()
df["sentiment"]=pd.to_numeric(df["sentiment"], errors="coerce")
df=df[df["sentiment"].isin([0,1])].copy()
df["sentiment"]=df["sentiment"].astype(int)


def clean_txt(text):
    #convert to lowercase
    text=str(text).lower()

    #remove punctuations and symbols
    text=re.sub(r"[^a-z0-9\s]", " ", text)

    #remove extra whitespace
    text=re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_txt"]=df["review"].apply(clean_txt)
df=df[df["clean_txt"] != ""].copy()

print("Original and Cleaned text:")
display(df[["review", "clean_txt"]].head(10))

print("Missing values after cleaning:")
print(df[["review", "sentiment", "clean_txt"]].isna().sum())
print("Exact duplicate cleaned reviews:", df["clean_txt"].duplicated().sum())

Original and Cleaned text:


,review,clean_txt
0,So there is no way for me to plug it in here i...,so there is no way for me to plug it in here i...
1,"Good case, Excellent value.",good case excellent value
2,Great for the jawbone.,great for the jawbone
3,Tied to charger for conversations lasting more...,tied to charger for conversations lasting more...
4,The mic is great.,the mic is great
5,I have to jiggle the plug to get it to line up...,i have to jiggle the plug to get it to line up...
6,If you have several dozen or several hundred c...,if you have several dozen or several hundred c...
7,If you are Razr owner...you must have this!,if you are razr owner you must have this
8,"Needless to say, I wasted my money.",needless to say i wasted my money
9,What a waste of money and time!.,what a waste of money and time


Missing values after cleaning:
review       0
sentiment    0
clean_txt    0
dtype: int64
Exact duplicate cleaned reviews: 28


Extra cleaning part - remove duplicates

In [13]:
starting_rows= len(df)
label_counts= df.groupby("clean_txt")["sentiment"].nunique()
conflicting_texts= set(label_counts[label_counts>1].index)

if conflicting_texts:
    df= df[df["clean_txt"].isin(conflicting_texts)].copy()

df= df.drop_duplicates(subset=["clean_txt"]).reset_index(drop=True)

print("Rows befor duplicate checks:", starting_rows)
print("Conflicting cleaned texts excluded:", len(conflicting_texts))
print("Rows after duplicate checks:", len(df))

Rows befor duplicate checks: 2720
Conflicting cleaned texts excluded: 0
Rows after duplicate checks: 2720


### 3. Explore text data

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Class distribution (0 = Negative, 1 = Positive):")
print(df["sentiment"].value_counts().sort_index())

print("Distribution across review sources:")
display(pd.crosstab(df["source"], df["sentiment"], rownames=["Source"], colnames=["Sentiment"]))

print("Example positive reviews:")
display(df.loc[df["sentiment"] == 1, ["review", "clean_txt"]].head(5))

print("Example negative reviews:")
display(df.loc[df["sentiment"] == 0, ["review", "clean_txt"]].head(5))

plt.figure(figsize=(6,4))
sns.countplot(data=df, x="sentiment", order=[0,1], color="tab:blue")
plt.xticks([0,1], ["Negative", "Positive"])
plt.xlabel("Sentiment")
plt.ylabel("Number of reviews")
plt.title("Sentiment Distribution")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "sentiment_distribution.png", dpi=150)
plt.show()